In [ ]:
import os
# Project data root. Override with the ACC_DATA_ROOT environment variable.
ACC_DATA_ROOT = os.environ.get("ACC_DATA_ROOT", os.path.expanduser("~/acc_data"))


In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
import scanpy as sc
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

In [ ]:
adata = sc.read_h5ad(f'{ACC_DATA_ROOT}/bulk_rna_data/h5ad_files/bulk_counts.h5ad')

In [ ]:
adata

In [ ]:
adata.obs.head()

In [ ]:
adata.obs['ClinicalOutcome'] = adata.obs['ClinicalOutcome'].map({'5Plus':'Good', '2Minus':'Poor'})
adata.obs['ModelOutcome2'] = adata.obs['ModelOutcome2'].map({'Average':'Good', 'Poor':'Poor'})
adata.obs['ModelOutcome1'] = adata.obs['ModelOutcome1'].map({'Average':'Good', 'Poor':'Poor'})

In [ ]:
adata.obs.rename(columns={
    'ClinicalOutcome':'Clinical Outcome',
    'DEGOutcome':'DEG Outcome',
    'ModelOutcome1':'ModelOutcome1',
    'ModelOutcome2':'Model Outcome'
}, inplace = True)

In [ ]:
# normalization and log transformation of expression values.
adata_tmp = adata.copy()
sc.pp.normalize_total(adata_tmp, target_sum=1e4)  # library-size normalize
sc.pp.log1p(adata_tmp)

In [ ]:
adata_tmp.var_names = adata_tmp.var['gene_label']

In [ ]:
# expression matrix: samples x genes
expr = pd.DataFrame(
    adata_tmp.X,
    index=adata_tmp.obs_names,
    columns=adata_tmp.var_names
)

In [ ]:
expr.head()

In [ ]:
# ---------------------------------------
# 2) Pick genes to show (DE-driven subset)
#    (padj<0.05, highest |log2FC|, and optionally baseMean filter)
# ---------------------------------------
de = adata.var.dropna(subset=["padj", "log2FoldChange"]).copy()

# Optional low-expression filter to avoid noisy genes
BASEMEAN_MIN = 10 # adjust as you like or set to 0 to disable
if "baseMean" in de.columns:
    de = de[de["baseMean"] >= BASEMEAN_MIN]

# choose top up/down (e.g., 25 each)
N = 20
up = de[(de["padj"] < 0.05) & (de["log2FoldChange"] > 0)].nsmallest(N, "padj")
dn = de[(de["padj"] < 0.05) & (de["log2FoldChange"] < 0)].nsmallest(N, "padj")

def names(df):
    if "gene_label" in df.columns:
        sym = df["gene_label"].astype(str)
        # Replace 'nan' with index
        return sym.where(sym != 'nan', df.index).astype(str)
    return df.index.astype(str)

genes = pd.Index(names(up).tolist() + names(dn).tolist())
genes = genes.intersection(expr.columns)  # keep only those present in expression

print(f"Found {len(up)} upregulated genes and {len(dn)} downregulated genes")
print(f"Total genes for heatmap (before adding genes of interest): {len(genes)}")

# ---------------------------------------
# Add genes of interest
# ---------------------------------------
genes_of_interest = [
    'MYB',
    'MYBL1',          
    'MYC',          
    'NOTCH1',
    'NFIB',
    'KIT',
    'EGFR',
    'TP53',
    'PTEN',
    'SOX4',
    'CDK6'
]

# Add genes of interest to the selection
genes_to_add = pd.Index(genes_of_interest).intersection(expr.columns)
genes = genes.union(genes_to_add)

print(f"Added {len(genes_to_add)} genes of interest: {', '.join(genes_to_add)}")
if len(genes_to_add) < len(genes_of_interest):
    missing = set(genes_of_interest) - set(genes_to_add)
    print(f"Warning: {len(missing)} genes not found in expression data: {', '.join(missing)}")
print(f"Total genes for heatmap: {len(genes)}")

# Identify which genes of interest are significantly DE
de_full = adata.var.dropna(subset=["padj", "log2FoldChange"]).copy()
sig_de_genes = set(names(de_full[(de_full["padj"] < 0.05)]))
non_sig_genes_of_interest = [g for g in genes_of_interest if g not in sig_de_genes and g in expr.columns]

print(f"Genes of interest that are NOT significantly DE: {', '.join(non_sig_genes_of_interest) if non_sig_genes_of_interest else 'None'}")

# ---------------------------------------
# 3) Build z-scored matrix (per gene)
# ---------------------------------------
mat = expr[genes]  # samples x selected genes
mat_z = (mat - mat.mean(axis=0)) / mat.std(axis=0)

# Drop any genes with NaN or infinite values after z-scoring
mat_z = mat_z.loc[:, ~mat_z.isnull().any(axis=0)]
mat_z = mat_z.loc[:, ~np.isinf(mat_z).any(axis=0)]

print(f"After removing NaN/Inf genes: {mat_z.shape[1]} genes remaining")

# Optional: order samples by metadata
# order_cols = [c for c in ["Model Outcome", "Clinical Outcome", "Site"] if c in adata.obs.columns]
order_cols = [c for c in ["Model Outcome", "Clinical Outcome"] if c in adata.obs.columns]
if order_cols:
    sample_order = adata.obs.sort_values(order_cols).index
    mat_z = mat_z.loc[sample_order]


# ---------------------------------------
# 4) Column annotations (sample bars) - Multiple annotations
# ---------------------------------------
# Define which columns to show as annotation bars
# annotation_cols = ['Site', 'Clinical Outcome', 'Model Outcome']
annotation_cols = ['Clinical Outcome', 'Model Outcome']

# Create color mappings for each annotation
col_colors_dict = {}

for col in annotation_cols:
    if col in adata.obs.columns:
        # Create a unique color palette for each annotation
        unique_vals = adata.obs[col].dropna().astype(str).unique()
        n_colors = len(unique_vals)
        
        # Use different color palettes for variety
        if col == 'Site':
            palette = sns.color_palette("tab10", n_colors=n_colors)
        elif col == 'Clinical Outcome':
            palette = sns.color_palette("Set2", n_colors=n_colors)
        else:  # Model Outcome
            palette = sns.color_palette("Set2", n_colors=n_colors)
        
        # Create color mapping
        lut = dict(zip(unique_vals, palette))
        col_colors_dict[col] = adata.obs.loc[mat_z.index, col].astype(str).map(lut)

# Convert to DataFrame
col_colors = pd.DataFrame(col_colors_dict) if col_colors_dict else None


# ---------------------------------------
# 5) Plot clustered heatmap with bold genes of interest and stars for non-sig genes
# ---------------------------------------
g = sns.clustermap(
    mat_z.T,                 # rows=genes, cols=samples
    cmap="RdBu_r", center=0,  # Deeper red and blue colors
    row_cluster=True, col_cluster=True,   # set to False to preserve your order
    col_colors=col_colors,
    xticklabels=True, yticklabels=True,  # Show sample names on x-axis
    figsize=(12, 16),
    vmin=-3, vmax=3,  # Clip extreme z-scores for better visualization
    cbar_pos=(0.1, 0.82, 0.01, 0.10),  # (x, y, width, height) - narrower and smaller colorbar
    cbar_kws={'label': 'Z-score', 'shrink': 0.5}
)

g.ax_heatmap.set_xlabel("Samples", fontsize=10)
g.ax_heatmap.set_ylabel("Genes (Z-score)", fontsize=10)
g.ax_heatmap.tick_params(axis='x', rotation=90, labelsize=8)  # Rotate sample names
# g.figure.suptitle(f"Top DE Genes + Genes of Interest Heatmap ({len(genes)} genes)", y=0.995, fontsize=12)

# ---------------------------------------
# Make genes of interest bold and add star if not significantly DE
# ---------------------------------------
# Get current labels
current_labels = [label.get_text() for label in g.ax_heatmap.get_yticklabels()]

# Create new labels with stars for non-significant genes of interest
new_labels = []
for gene_name in current_labels:
    if gene_name in non_sig_genes_of_interest:
        new_labels.append(f"{gene_name} *")
    else:
        new_labels.append(gene_name)

# Set the new labels
g.ax_heatmap.set_yticklabels(new_labels)

# Now make genes of interest bold
for label in g.ax_heatmap.get_yticklabels():
    gene_name_check = label.get_text().replace(' *', '')  # Remove star for checking
    if gene_name_check in genes_of_interest:
        label.set_weight('bold')
        label.set_fontsize(9)

# ---------------------------------------
# Add legends for all annotation bars - side by side at the top
# ---------------------------------------
if col_colors is not None:
    from matplotlib.patches import Patch
    
    # Create handles and labels for all annotations
    all_handles = []
    all_labels = []
    
    for title in annotation_cols:
        if title in col_colors_dict:
            # Get unique values from the original data
            unique_vals = adata.obs.loc[mat_z.index, title].astype(str).unique()
            
            # Get colors for each unique value
            for val in unique_vals:
                # Find the color for this value
                color = col_colors_dict[title][adata.obs.loc[mat_z.index, title].astype(str) == val].iloc[0]
                all_handles.append(Patch(facecolor=color, edgecolor='black', linewidth=0.5))
                all_labels.append(f"{title}: {val}")
    
    # Add a single legend with all annotations
    g.fig.legend(
        handles=all_handles,
        labels=all_labels,
        loc='upper center',
        bbox_to_anchor=(0.75, 1),
        ncol=1,
        frameon=True,
        fontsize=8,
        title_fontsize=9
    )

plt.tight_layout()
# plt.savefig(f'{ACC_DATA_ROOT}/bulk_rna_data/outputs/images/DEG_heatmap.svg',
#             format = 'svg',
#             dpi = 600,
#             bbox_inches = 'tight')

plt.show()
plt.close()

In [ ]:
# # Extract top 30 upregulated and top 30 downregulated genes
# de_genes = adata.var.dropna(subset=["padj", "log2FoldChange"]).copy()

# # Optional: apply baseMean filter
# BASEMEAN_MIN = 10
# if "baseMean" in de_genes.columns:
#     de_genes = de_genes[de_genes["baseMean"] >= BASEMEAN_MIN]

# # Get top 30 upregulated genes (positive log2FC, lowest padj)
# top30_up = de_genes[(de_genes["padj"] < 0.05) & (de_genes["log2FoldChange"] > 0)].nsmallest(30, "padj")

# # Get top 30 downregulated genes (negative log2FC, lowest padj)
# top30_down = de_genes[(de_genes["padj"] < 0.05) & (de_genes["log2FoldChange"] < 0)].nsmallest(30, "padj")

# # Display upregulated genes
# print("=" * 80)
# print("TOP 30 UPREGULATED GENES")
# print("=" * 80)
# print(f"\nFound {len(top30_up)} upregulated genes\n")
# up_display = top30_up[["Symbol", "log2FoldChange", "padj", "baseMean"]].copy()
# up_display["Symbol"] = up_display["Symbol"].astype(str)
# print(up_display.to_string())

# # Display downregulated genes
# print("\n" + "=" * 80)
# print("TOP 30 DOWNREGULATED GENES")
# print("=" * 80)
# print(f"\nFound {len(top30_down)} downregulated genes\n")
# down_display = top30_down[["Symbol", "log2FoldChange", "padj", "baseMean"]].copy()
# down_display["Symbol"] = down_display["Symbol"].astype(str)
# print(down_display.to_string())

# # Optional: Save to CSV files
# top30_up.to_csv(f'{ACC_DATA_ROOT}/bulk_rna_data/top30_upregulated_genes.csv')
# top30_down.to_csv(f'{ACC_DATA_ROOT}/bulk_rna_data/top30_downregulated_genes.csv')
# print("\n" + "=" * 80)
# print("Files saved:")
# print("  - top30_upregulated_genes.csv")
# print("  - top30_downregulated_genes.csv")
# print("=" * 80)